# ST 554 - Project 2
Author: Max Campbell

## Part I - Creating a Class

In object-oriented programming, one of the foundational components of a script is a class. A simple definition of a class is that it is a framework from which we can create objects. For our purposes, objects are instances of a class, which we can manipulate using methods defined within the class. Let's take a closer look at a class to understand the powerful functionality that we can apply to our work in big data.

The first thing we need to do is write a python script with the class contained within! This includes defining the class itself, as well as the key attributes that we want to be able to access in an instance of the class. To keep things simple, we'll use just one attribute, a `pyspark` DataFrame. The full script for our example class with annotated code, `SparkDataCheck`, can be found in the "SparkDataCheck.py" file. Let's go ahead and import the modules we will need.

In [1]:
#Import/reload modules
import SparkDataCheck
from pyspark.sql import SparkSession
import pandas as pd
import importlib
importlib.reload(SparkDataCheck)

<module 'SparkDataCheck' from '/home/jupyter-mscampb4@ncsu.edu/Project2/SparkDataCheck.py'>

As we mentioned previously, we are using a `pyspark` DataFrame as the main piece of data, stored in an attribute of the class that is defined whenever we instantiate it. This means that we should create a local Spark session, then use it to import a comma-separated values (CSV) file as our DataFrame.

In [60]:
#Instantiate SparkDataCheck using air.csv
spark = SparkSession.builder.master('local[*]').appName('Project2').config("spark.sql.ansi.enabled", "false").getOrCreate() #Initialize pyspark session
air = SparkDataCheck.SparkDataCheck.load_pyspark(spark, "air.csv")
air.df.show()

+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|          948|    172|        1092|    122|        1584| 

26/03/22 10:07:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-mscampb4@ncsu.edu/Project2/air.csv


Let's take a closer look at the second line of code in the section above. First, we call `SparkDataCheck`, which is the module we previously imported. From there, we can call on our class `SparkDataCheck` to reference it directly. This creates a `SparkDataCheck` object that accepts one argument: the `pyspark` DataFrame we just imported. However, we can see that we called a function that looks something like `load_pyspark(SparkSession, string)`. This is a class method! Denoted by the `@classmethod` tag in our script, a class method is a method that can be run when the object is first created to directly modify any attributes. Class methods are extremely helpful when there are multiple ways to handle data that we want to support. For example, we created the DataFrame directly from a CSV file, but what if we wanted to create it using `pandas` instead?

In [3]:
#Alternatively, load in using a pandas df and initialize the class in this manner
airPd = pd.read_csv("air.csv")

#Do some data cleaning while we are here
airPd = airPd.replace([-200], [None])
airPd['Date'] = airPd.Date.astype('string')
airPd = airPd.rename(columns={'PT08.S1(CO)': 'PT08S1(CO)', 
                      'PT08.S2(NMHC)': 'PT08S2(NMHC)', 
                      'PT08.S3(NOx)': 'PT08S3(NOx)',
                      'PT08.S4(NO2)': 'PT08S4(NO2)',
                      'PT08.S5(O3)': 'PT08S5(O3)'}) #Removing the dots as they lead to errors

#Load it into pyspark
air = SparkDataCheck.SparkDataCheck.load_pandas(spark, airPd)

We instantiated the SparkDataCheck object using the same general format, but this time the class method was `load_pandas(SparkSession, pandas dataframe)`. Both methods arrived at the same destination, but used different types of underlying data structures to get there. Now that we have an instance of SparkDataCheck, we can use methods to manipulate it! A normal method is different from a class method in that it have to run when the class is first created. This means that we can call regular methods on an existing instance of our object. In `SparkDataCheck`, we've created a few methods that validate some aspects of the DataFrame, as well as a couple methods that generate summary statistics. Let's go through them one-by-one.

First, we have the `validate_numeric()` method. This method accepts a column as input, and returns our DataFrame with an appended column of Boolean values denoting whether the value in the specified column falls within a set of bounds that we can define as well. There are several different behaviors that we want to support. First, let's make sure we've defined what happens when a non-numeric column is input:

In [4]:
#Validate various columns for numeric values
air.validate_numeric("Date", lower = 0, upper = 100).show() #Returns error due to incorrect col type

Please provide a numeric column.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|      

We've printed out a message, and then returned the initial DataFrame. Now, let's see what happens if the inputs are as expected.

In [5]:
air.validate_numeric("CO(GT)", lower = 0, upper = 100).show() #Evaluates

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_num_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                 true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                 true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

We've appended a column with boolean values. This is what we wanted! Notice that when the value in the user-specified column is NULL, the new column also returns a NULL value. This is expected behavior for our purposes, as we don't want to evaluate whether a value falls within certain bounds if it doesn't exist, after all. Next up, what if the user only wants to evaluate against one bound?

In [6]:
air.validate_numeric("CO(GT)", lower = 1).show() #Evaluates
air.validate_numeric("CO(GT)", upper = 1.5).show() #Evaluates

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+-----------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+-----------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|             true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|             true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|             true

Looks like both versions of this method call are working as intended! By accounting for this behavior, we've created more use cases for the method, which will benefit us in further analysis. Lastly, what happens if no bounds are provided?

In [7]:
air.validate_numeric("CO(GT)").show() #Returns error due to lack of bounds

Please provide at least one bound.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|    

This results in another "error" message, which is reasonable.

Let's also look at a similar method for string-based columns. For this method, we accept a user-specified column and a list of valid "levels", and return a boolean column appended to our DataFrame, similar to the previous method. There are two distinct behaviors in this method. The first of which occurs when a non-string-based column is input as an argument:

In [8]:
#Validate various columns for string values
validLevels = ["3/10/2004"]
air.validate_string("CO(GT)", validLevels).show() #Error due to ineligible data type

Please provide a string column.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|       

This is similar to the behavior in the previous method, which is good because it is consistent. Now, let's look at a valid call of the method:

In [9]:
air.validate_string("Date", validLevels).show() #evaluates as expected

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_str_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                 true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                 true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

Likewise, this is consistent with the previous method.

In [10]:
#A few more column arguments, for good measure.
air.validate_string("Time", validLevels).show() #evaluates to all false because no valid levels should occur
air.validate_string("Date", None).show() #NULL should evaluate to NULL

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_str_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                false|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                false|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

The last validation method that we have is simple. The user will specify a column, and the DataFrame will return with Boolean values appended to it indicating whether a value in that column was NULL or not. This method is not picky, so there's really only one behavior defined:

In [11]:
#Test denote_null_values method, all calls should evaluate under expected behavior
air.denote_null_values("CO(GT)").show()
air.denote_null_values("NOx(GT)").show()
air.denote_null_values("NO2(GT)").show()
air.denote_null_values("T").show()

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+--------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|has_null_value|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+--------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|         false|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|         false|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|         false|
|         3|3/10

Looks like this is working as expected! This method can be useful for filtering out missing values from a dataset before doing some more exhaustive analysis on it. 

Speaking of doing some more exhaustive analysis, let's take a look at the summarization methods that were created with this class. The first is `get_min_max()`, which simply accepts a user-specified column (and an optional grouping column), and returns the minimum and maximum values for each applicable group (whether it be a whole column, or those within a group defined by the grouping column). The benefits of creating this method include being able to seamlessly handle a grouping column without asking the user to augment their program significantly. Note that the remaining methods all return `pandas` dataframes. Let's check out some examples!

In [12]:
#Test get_min_max
air.get_min_max(col = "CO(GT)") #Returns a dataframe with one row

,min(CO(GT)),max(CO(GT))
0,0.1,11.9


Shown above is the simplest iteration of this method call, which evaluated across an entire column. Now, let's take a look at what happens when we start experimenting with grouping variables.

In [13]:
air.get_min_max(col = "CO(GT)", groupCol = "Date").head() #Returns a dataframe with one row per date

,Date,min(CO(GT)),max(CO(GT))
0,3/11/2004,0.6,6.9
1,3/12/2004,0.6,6.6
2,3/10/2004,1.2,2.6
3,3/13/2004,1.0,4.2
4,3/15/2004,1.0,8.1


Above is the minimum and maximum for a given date. We see that we have one minimum and one maximum value per row, which is an intuitive format for reading this data. Now let's see what happens if we forget (or decide) to leave out the column that we want to evaluate.

In [14]:
air.get_min_max() #Returns a minimum and maximum for every numeric column

26/03/22 09:05:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,min(Unnamed: 0),min(CO(GT)),min(PT08S1(CO)),min(NMHC(GT)),min(C6H6(GT)),min(PT08S2(NMHC)),min(NOx(GT)),min(PT08S3(NOx)),min(NO2(GT)),min(PT08S4(NO2)),...,max(C6H6(GT)),max(PT08S2(NMHC)),max(NOx(GT)),max(PT08S3(NOx)),max(NO2(GT)),max(PT08S4(NO2)),max(PT08S5(O3)),max(T),max(RH),max(AH)
0,0,0.1,647,7,0.1,383,2,322,2,551,...,63.7,2214,1479,2683,340,2775,2523,44.6,88.7,2.231


Looks like we get a minimum and maximum for every numeric column in the DataFrame! We had several options for what we could do here, but choosing this behavior makes sense as the user still gets an evaluation and can use it to inform future method calls or potential edits to their program. Let's see what happens if we specify a grouping variable, but no column to evaluate.

In [15]:
air.get_min_max(groupCol = "Date").head() #Returns a minimum and maximum for every numeric column with one row per date

,Date,min(Unnamed: 0),min(CO(GT)),min(PT08S1(CO)),min(NMHC(GT)),min(C6H6(GT)),min(PT08S2(NMHC)),min(NOx(GT)),min(PT08S3(NOx)),min(NO2(GT)),...,max(C6H6(GT)),max(PT08S2(NMHC)),max(NOx(GT)),max(PT08S3(NOx)),max(NO2(GT)),max(PT08S4(NO2)),max(PT08S5(O3)),max(T),max(RH),max(AH)
0,3/11/2004,6,0.6,913.0,8.0,1.1,512.0,16.0,702.0,28.0,...,27.4,1488.0,383.0,1918.0,172.0,2333.0,1704.0,11.3,81.1,0.8778
1,3/12/2004,30,0.6,831.0,7.0,1.0,501.0,21.0,624.0,32.0,...,32.6,1610.0,340.0,1895.0,170.0,2390.0,1887.0,16.9,65.9,0.7771
2,3/10/2004,0,1.2,1197.0,38.0,4.7,750.0,89.0,1056.0,92.0,...,11.9,1046.0,172.0,1337.0,122.0,1692.0,1268.0,13.6,60.0,0.7888
3,3/13/2004,54,1.0,978.0,27.0,2.6,625.0,53.0,754.0,60.0,...,19.6,1286.0,296.0,1420.0,165.0,1922.0,1886.0,19.4,71.9,0.8193
4,3/15/2004,102,1.0,1075.0,39.0,3.9,703.0,66.0,537.0,71.0,...,39.2,1754.0,478.0,1156.0,187.0,2679.0,2184.0,24.4,70.5,1.0945


It behaves similarly to the previous two calls, where there is one row per date, and all the numeric columns are evaluated. Once again, this is consistent with previous behaviors we have demonstrated. Lastly, let's specify a non-numeric column for the evaluation and see what happens.

In [16]:
air.get_min_max(col = "Date") #Returns an error due to incorrect data type

Please provide a numeric column.


DataFrame[Unnamed: 0: bigint, Date: string, Time: string, CO(GT): double, PT08S1(CO): bigint, NMHC(GT): bigint, C6H6(GT): double, PT08S2(NMHC): bigint, NOx(GT): bigint, PT08S3(NOx): bigint, NO2(GT): bigint, PT08S4(NO2): bigint, PT08S5(O3): bigint, T: double, RH: double, AH: double]

We've returned a string message asking the user to provide a numeric column. With this, it looks like we've accounted for most of the user behavior that we expect with this function!

The last method that was created for `SparkDataCheck` is the `get_count()` function. This function essentially generates a 1-or-2-way frequency table using user-specified inputs for both variables. The important caveat with this function is that we need both variables to be string-based data types. Let's take a closer look:

In [17]:
#Test get_count
air.get_count(col = "CO(GT)") #Returns error due to non-string data type
air.get_count(col = "Date", groupCol = "CO(GT)") #Returns an error due to non-string data type

Please provide a string column.
Please provide a string column for the grouping variable.


DataFrame[Unnamed: 0: bigint, Date: string, Time: string, CO(GT): double, PT08S1(CO): bigint, NMHC(GT): bigint, C6H6(GT): double, PT08S2(NMHC): bigint, NOx(GT): bigint, PT08S3(NOx): bigint, NO2(GT): bigint, PT08S4(NO2): bigint, PT08S5(O3): bigint, T: double, RH: double, AH: double]

We've created two defined error behaviors, one for if the first column is non-string-based, and another for the grouping column. This shows that a message will return if either column doesn't meet the requirements, which is expected.

In [18]:
air.get_count(col = "Date").head() #Gets number of observations per date

,Date,count(Date)
0,3/11/2004,24
1,3/12/2004,24
2,3/10/2004,6
3,3/13/2004,24
4,3/15/2004,24


Above is a demonstration of a 1-way frequency table, counting observations by date. Now, let's look at the 2-way table:

In [19]:
air.get_count(col = "Date", groupCol = "Time").head() #Gets number of observations per date/time combo

,Date,Time,count(Date)
0,3/12/2004,11:00:00,1
1,3/11/2004,21:00:00,1
2,3/13/2004,0:00:00,1
3,3/12/2004,7:00:00,1
4,3/11/2004,18:00:00,1


While the practicality of this particular method call is questionable (since each date/time combination should be unique in this particular context), we see that it did generate a two-way count of all the observations. This means our function is working as intended! With this, we've now created a class and verified a set of expected behaviors that are possible with the methods we've created. As such, the `SparkDataCheck` class is ready for some applied usage!

## Part II - NFL Weekly Data using Spark

In this part, we will explore the flexibility that spark offers in terms of data exploraton. Specifically, we can leverage spark in multiple different ways! The first method we will use involves `pandas`-on-Spark, which is rooted in the `pandas` framework, as you might have guessed. For this demonstration, we will use a dataset of weekly NFL scoring data. Let's go ahead and read everything in:

In [45]:
#Import necessary modules (pandas was loaded in part 1)
import numpy as np
import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False) #Prevents an error when converted from pandas dataframe

#Read in NFL data
nfl_pd = pd.read_csv("weekly_nfl_data.csv")
nfl = ps.from_pandas(nfl_pd)
nfl.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


That's a lot of columns! Let's look at the schema to see what we're working with:

In [26]:
print(nfl.columns)

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

We're mainly interested in passing statistics for quarterbacks (QBs) in the regular season between 2005 and 2023, so let's go ahead and subset the dataset as needed.

In [30]:
#Subset to the guidelines described above
nfl_sub = nfl.loc[(nfl['position'] == "QB") & (nfl['season_type'] == "REG") & (nfl['season'] >= 2005) & (nfl['season'] <= 2023), 
                  ['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']]
nfl_sub.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Now we can do some summary statistics for our variables of interest.

In [38]:
#Get the sum and the mean of each grouping of player/season.
nfl_group = nfl_sub.groupby(['player_display_name', 'season'])[['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']].agg(['sum','mean'])
nfl_group.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333

There are two valuable quarterback statistics that we can create from these variables: the completion percentage, which is a ratio of a player's passes that their receiver caught against their total number of passing attempts; and the touchdown-interception ratio, which is somewhat self-explanatory in that it measures a player's passing touchdowns against their interceptions thrown.

In [53]:
#Create completion_percentage and td_int_ratio columns
nfl_group['completion_percentage'] = nfl_group['completions']['sum'] / nfl_group['attempts']['sum']
nfl_group['td_int_ratio'] = nfl_group['passing_tds']['sum'] / nfl_group['interceptions']['sum']
nfl_group.head()

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000              0.888889          inf
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000              0.560000     0.500000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333              0.638889     1.000000

We can see that some of these values will be a bit wonky for those that didn't play a lot during their time in the NFL. As such, we will subset this dataset by those who attempted at least 50 passes in a given season.

In [55]:
#Subset by > 50 attempts in a season
nfl_group_att = nfl_group.loc[nfl_group['attempts']['sum'] >= 50, :]
nfl_group_att.head()

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Jeff Garcia         2005           102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000              0.589595     0.500000
Brett Favre         2005           372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500              0.612850     0.689655
Todd Bouman         2005            68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000              0.557377     0.285714

This should hopefully insulate us from the weirder results a little bit. Now that we are looking at quarterbacks with a solid sample size, let's see who performed the best over a given season!

In [58]:
#Sort descening completion_percentage
nfl_group_att.sort_values(by = 'completion_percentage', ascending = False).head(40)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
C.J. Beathard       2023            40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Colt McCoy          2021            74  10.571429       99  14.142857         740.0  105.714286           3  0.428571           1.0  0.142857              0.747475     3.000000
Matt Schaub         2019            50  10.000000       67  13.400000         580.0  116.000000           3  0.600000           1.0  0.200000              0.746269     3.000000
Drew Brees          2018           364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333              0.744376     6.400000
                    2019           281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636              0.743386     6.750000
Mason Rudolph       2023            55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Taysom Hill         2020            88   5.500000      121   7.562500         928.0   58.000000           4  0.250000           2.0  0.125000              0.727273     2.000000
Nick Foles          2018           141  28.200000      195  39.000000        1413.0  282.600000           7  1.400000           4.0  0.800000              0.723077     1.750000
Drew Brees          2017           386  24.125000      536  33.500000        4334.0  270.875000          23  1.437500           8.0  0.500000              0.720149     2.875000
Sam Bradford        2016           395  26.333333      552  36.800000        3877.0  258.466667          20  1.333333           5.0  0.333333              0.715580     4.000000
Drew Brees          2011           471  29.437500      660  41.250000        5535.0  345.937500          46  2.875000          14.0  0.875000              0.713636     3.285714
Colt McCoy          2014            91  18.200000      128  25.600000        1057.0  211.400000           4  0.800000           3.0  0.600000              0.710938     1.333333
Aaron Rodgers       2020           372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500              0.707224     9.600000
Bailey Zappe        2022            65  16.250000       92  23.000000         781.0  195.250000           5  1.250000           3.0  0.750000              0.706522     1.666667
Drew Brees          2009           363  24.200000      514  34.266667        4388.0  292.533333          34  2.266667          11.0  0.733333              0.706226     3.090909
                    2020           275  22.916667      390  32.500000        2942.0  245.166667          24  2.000000           6.0  0.500000              0.705128     4.000000
Joe Burrow          2021           366  22.875000      520  32.500000        4611.0  288.187500          34  2.125000          14.0  0.875000              0.703846     2.428571
Derek Carr          2019           361  22.562500      513  32.062500        4054.0  253.375000          21  1.312500           8.0  0.500000              0.703704     2.625000
Jake Browning       2023           171  19.000000      243  27.000000        1936.0  215.111111          12  1.333333           7.0  0.777778              0.703704     1.714286
Chase Daniel        2019            45  15.000000       64  21.333333         435.0  145.000000           3  1.000000           2.0  

In [59]:
#Sort descening td_int_ratio
nfl_group_att.sort_values(by = 'td_int_ratio', ascending = False).head(40)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Matt Schaub         2005            33   4.714286       64   9.142857         495.0   70.714286           4  0.571429           0.0  0.000000              0.515625          inf
Charlie Batch       2006            30   4.285714       52   7.428571         477.0   68.142857           5  0.714286           0.0  0.000000              0.576923          inf
Todd Collins        2007            67  16.750000      105  26.250000         888.0  222.000000           5  1.250000           0.0  0.000000              0.638095          inf
Troy Smith          2007            40  10.000000       76  19.000000         452.0  113.000000           2  0.500000           0.0  0.000000              0.526316          inf
Jake Locker         2011            34   6.800000       66  13.200000         542.0  108.400000           4  0.800000           0.0  0.000000              0.515152          inf
Derek Anderson      2014            65  13.000000       97  19.400000         701.0  140.200000           5  1.000000           0.0  0.000000              0.670103          inf
Brian Hoyer         2016           134  22.333333      200  33.333333        1445.0  240.833333           6  1.000000           0.0  0.000000              0.670000          inf
Nick Foles          2016            36  18.000000       55  27.500000         410.0  205.000000           3  1.500000           0.0  0.000000              0.654545          inf
Jimmy Garoppolo     2016            43   7.166667       63  10.500000         502.0   83.666667           4  0.666667           0.0  0.000000              0.682540          inf
Matt Moore          2019            59   9.833333       91  15.166667         659.0  109.833333           4  0.666667           0.0  0.000000              0.648352          inf
C.J. Beathard       2020            66  11.000000      104  17.333333         787.0  131.166667           6  1.000000           0.0  0.000000              0.634615          inf
Desmond Ridder      2022            73  18.250000      115  28.750000         708.0  177.000000           2  0.500000           0.0  0.000000              0.634783          inf
Andy Dalton         2023            34  17.000000       58  29.000000         361.0  180.500000           2  1.000000           0.0  0.000000              0.586207          inf
C.J. Beathard       2023            40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Mason Rudolph       2023            55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Tom Brady           2016           291  24.250000      432  36.000000        3554.0  296.166667          28  2.333333           2.0  0.166667              0.673611    14.000000
Nick Foles          2013           203  15.615385      317  24.384615        2891.0  222.384615          27  2.076923           2.0  0.153846              0.640379    13.500000
Josh McCown         2013           149  18.625000      224  28.000000        1829.0  228.625000          13  1.625000           1.0  0.125000              0.665179    13.000000
Aaron Rodgers       2018           372  23.250000      597  37.312500        4442.0  277.625000          25  1.562500           2.0  0.125000              0.623116    12.500000
Damon Huard         2006           148  14.800000      244  24.400000        1878.0  187.800000          11  1.100000           1.0  

Those with `inf` touchdown/interception ratios didn't throw a single interception during the regular season! That's impressive. We also see that Tom Brady put up some ridiculous numbers, including a TD/Interception ratio of 14 in the 2016 regular season. 

Now that we've arrived at some usable data, let's backtrack a bit and perform the same operations using Spark SQL. We want to demonstrate the flexibility of Spark as a tool for data science, so it follows that we should be able to take two different routes to arrive at the same conclusion.

In [66]:
#Import necessary modules
from pyspark.sql import functions as F

#Re-create the above steps, using Spark SQL
#Note that we already have a SparkSession from part 1, we will use it here.
#Below is the code that would be necessary to create the SparkSession if necessary.
#spark = SparkSession.builder.master('local[*]').appName('Project2').getOrCreate()

#Read in NFL data
nfl = spark.read.load("weekly_nfl_data.csv",
                      format = "csv",
                      sep = ",",
                      inferSchema = "true",
                      header = "true")
nfl.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [77]:
#Show columns attribute
nfl.columns

<bound method DataFrame.printSchema of DataFrame[player_id: string, player_name: string, player_display_name: string, position: string, position_group: string, headshot_url: string, recent_team: string, season: int, week: int, season_type: string, opponent_team: string, completions: int, attempts: int, passing_yards: double, passing_tds: int, interceptions: double, sacks: double, sack_yards: double, sack_fumbles: int, sack_fumbles_lost: int, passing_air_yards: double, passing_yards_after_catch: double, passing_first_downs: double, passing_epa: double, passing_2pt_conversions: int, pacr: double, dakota: double, carries: int, rushing_yards: double, rushing_tds: int, rushing_fumbles: double, rushing_fumbles_lost: double, rushing_first_downs: double, rushing_epa: double, rushing_2pt_conversions: int, receptions: int, targets: int, receiving_yards: double, receiving_tds: int, receiving_fumbles: double, receiving_fumbles_lost: double, receiving_air_yards: double, receiving_yards_after_catch:

There are some extra settings we need when reading in the data itself, but we can see that the `columns` attribute in both `pandas`-on-Spark and Spark SQL behave similarly.

In [71]:
#Perform the same subset as before: QB passing stats in the regular season between 2005-2023.
nfl_sub = nfl.select('player_display_name', 'season', 'week', 'season_type', 'position', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions') \
        .filter((F.col('position') == 'QB') & (F.col('season_type') == 'REG') & (F.col('season') >= 2005) & (F.col('season') <= 2023))
nfl_sub.show(5)

+-------------------+------+----+-----------+--------+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|season_type|position|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|        REG|      QB|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|        REG|      QB|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|        REG|      QB|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|        REG|      QB|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|        REG|      QB|          1|       1|         11.0|          0|          0.0|
+-------------------+------+----+-----------+--------+-----------+------

In [85]:
#Calculate sum and mean
nfl_group = nfl_sub.select('player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions') \
            .groupBy('player_display_name', 'season') \
            .agg(F.sum('completions'), F.sum('attempts'), F.sum('passing_yards'), F.sum('passing_tds'), F.sum('interceptions'),
                 F.avg('completions'), F.avg('attempts'), F.avg('passing_yards'), F.avg('passing_tds'), F.avg('interceptions'))
nfl_group.show(5)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006|             263|          431|            2805.0|              17|              11.0| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|
|       Jake Plummer|  2005|             277|          456|            3366.0|              18|               7.0|           17.3125|              28.5|        

One notable difference we've come across is that in the `pandas`-on-Spark implementation of our DataFrame, we used nested column indexes to access specific columns, such as the sum of the completions per season. In the Spark SQL implementation, we don't have any nested column names. This doesn't create any major changes for us, but it does look notably different when viewing the DataFrame.

In [88]:
#Create completion_percentage and td_int_ratio
nfl_stats = nfl_group.select('*') \
            .withColumn("completion_percentage", (nfl_group['sum(completions)'] / nfl_group['sum(attempts)'])) \
            .withColumn("td_int_ratio", (nfl_group['sum(passing_tds)'] / nfl_group['sum(interceptions)']))

nfl_stats.show(5)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|             263|          431|            2805.0|              17|              11.0| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|   0.6102088167053364|1.54545454545454

Now we have come across our first impactful difference in how the two methods are implemented. Specifically, Spark SQL handles division by zero by returning a NULL value. `pandas`-on-Spark chose to return a value of `inf`. As a result, the sorting operation we are about to do will likely look different when we sort for the highest touchdown/interception ratios. 

In [90]:
#Sort by completion percentage and display top 40 values
nfl_stats.select('*') \
         .filter(nfl_group['sum(attempts)'] >= 50) \
         .sort(nfl_stats.completion_percentage, ascending = False).show(40)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|  avg(completions)|     avg(attempts)|avg(passing_yards)|   avg(passing_tds)| avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|              40|           53|             349.0|               1|               0.0| 6.666666666666667| 8.833333333333334|58.166666666666664|0.16666666666666666|                0.0|   0.7547169811320755|        

In [91]:
#Sort by td_int_ratio and display top 40 values
nfl_stats.select('*') \
         .filter(nfl_group['sum(attempts)'] >= 50) \
         .sort(nfl_stats.td_int_ratio, ascending = False).show(40)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+-----------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)| avg(interceptions)|completion_percentage|     td_int_ratio|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+-----------------+
|          Tom Brady|  2016|             291|          432|            3554.0|              28|               2.0|             24.25|              36.0| 296.1666666666667|2.3333333333333335|0.16666666666666666|   0.6736111111111112|             14

Much like we theorized, the NULL values that occurred when a player didn't throw an interception in a given season were bumped off of the top. This moves Tom Brady's 2016 season up to the top as a result. There are merits to considering both implementations of this ratio: on one hand, throwing zero interceptions in a season can be considered an accomplishment that some may consider worthy of being at the top of these lists; on the other hand, some seasons like Tom Brady in 2016 or Nick Foles in 2013 had significantly higher volumes of passing touchdowns than some of the zero interception seasons we observed in our previous implementation (the highest was accomplished by CJ Beathard in 2020 with zero interceptions and six passing touchdowns). 

A fair compromise in cases like this might be considering the difference between passing touchdowns and interceptions in lieu of a ratio, but I digress. We can see that the Spark framework has provided us with two distinct ways to handle large datasets, and both appear equally viable in accomplishing the tasks that we set out to do. There will always be minor differences in behavior/implementation, but learning these differences and understanding the decisions behind these choices can inform us in future work and allow us to pick the implementation that best suits the data we are working with!